In [ ]:
import os
import subprocess
from pathlib import Path
import rasterio
from rasterio.warp import transform_bounds
import seamless_3dep as s3dep
import whitebox

# ==========================================
# 1. Read exact grid geometry from NLCD Master
# ==========================================
nlcd_master_path = "/kaggle/input/datasets/maanav0114/ambystoma-sdm-data/land_cover_clipped_reproj.tiff"

with rasterio.open(nlcd_master_path) as master:
    master_crs = master.crs.to_string()
    minx, miny, maxx, maxy = master.bounds
    width = master.width
    height = master.height

# Convert the NLCD bounds (in meters) to standard Lon/Lat (EPSG:4326) for downloading
# Pad by ~1000m just to ensure the downloaded tiles fully cover the warped corners
pad = 1000.0
bbox_4326 = transform_bounds(
    master_crs, "EPSG:4326", 
    minx - pad, miny - pad, maxx + pad, maxy + pad
)


In [ ]:
# ==========================================
# 2. Download 30m DEM tiles to disk
# ==========================================
work_dir = Path("/kaggle/working/terrain_data")
tile_dir = work_dir / "dem_tiles"
tile_dir.mkdir(parents=True, exist_ok=True)

# Absolute WhiteboxTools output paths so nothing is written relative to the CWD
dem_breached = str(work_dir / "dem_breached.tif")
slope_deg = str(work_dir / "slope_deg.tif")
sca_file = str(work_dir / "sca.tif")
final_twi = str(work_dir / "ohio_region_twi_aligned.tif")

print("Downloading 30m DEM tiles from USGS 3DEP...")
tiff_files = s3dep.get_dem(bbox_4326, tile_dir, res=30, buff_npixels=2)

# Build a lightweight virtual mosaic (.vrt) of the downloaded tiles
vrt_file = str(work_dir / "merged_4326.vrt")
s3dep.build_vrt(vrt_file, tiff_files)


In [ ]:
# ==========================================
# 3. Snap & Warp DEM directly to NLCD Master Grid
# ==========================================
aligned_dem = str(work_dir / "ohio_region_dem_aligned.tif")

print("Warping and locking DEM onto the NLCD Master pixel grid...")
subprocess.run([
    "gdalwarp",
    "-t_srs", master_crs,
    "-te", str(minx), str(miny), str(maxx), str(maxy),
    "-ts", str(width), str(height),
    "-r", "bilinear",
    "-dstnodata", "-9999.0",
    "-co", "COMPRESS=LZW",
    "-co", "BIGTIFF=YES",
    "-multi", "-wo", "NUM_THREADS=ALL_CPUS",
    "-overwrite",
    vrt_file,
    aligned_dem
], check=True)


In [ ]:
# ==========================================
# 4. Run WhiteboxTools on the Aligned DEM
# ==========================================
wbt = whitebox.WhiteboxTools()
wbt.set_working_dir(str(work_dir))

print("Conditioning DEM and calculating TWI...")
# Step 4a: Breach small artificial road/noise dams (max 60m / 2 pixels)
wbt.breach_depressions_least_cost(
    dem=aligned_dem, 
    output=dem_breached, 
    dist=60.0, 
    fill=True
)

# Step 4b: Calculate Slope in degrees
wbt.slope(
    dem=dem_breached, 
    output=slope_deg, 
    units="degrees"
)

# Step 4c: Calculate Specific Catchment Area (D-Infinity)
wbt.d_inf_flow_accumulation(
    i=dem_breached, 
    output=sca_file, 
    out_type="Specific Contributing Area"
)

# Step 4d: Compute final TWI (inherits exact alignment from ohio_region_dem_aligned.tif)
wbt.wetness_index(
    sca=sca_file, 
    slope=slope_deg, 
    output=final_twi
)

print(f"Done! Final aligned TWI saved to: {final_twi}")